<p> <center> <a href="../../Start-NIM-RAG.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="rag_nim_endpoints.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="rag_nim_endpoints.ipynb">1</a>
        <a >2</a>
        <a href="nim_lora_adapter.ipynb">3</a>
        <!-- <a href="challenge.ipynb">4</a> -->
    </span>
    <span style="float: left; width: 33%; text-align: right;"><a href="nim_lora_adapter.ipynb">Next Notebook</a></span>
</div>

# ローカライズされたNVIDIA NIMとGoogle Kubernetes EngineによるRAGの構築
---


このNotebookでは、ローカライズされたNVIDIA NIM(NVIDIA Inference Microservice)とGKE(Google Kubernetes Engine)を使用してRAG (検索拡張生成)パイプラインを構築するデモンストレーションを行います。まず初めに、RAGの概要およびNVIDIA API Keyの設定、NIMイメージの取得とデプロイ、ローカルにデプロイされたNIMを使用するRAGアプリケーションの構築について説明します。

- NIMモデルにアクセスするためのNVIDIA APIキーの設定方法
- RAGのコンセプトの理解
- GKE(Google Kubernetes Engine)の概要およびNVIDIA NIMのGKE経由でのデプロイの方法
- ウェブリンクからコンテンツを取得し、コンテンツをテキストチャンクに分割し、エンベッディングを作成し、ベクターストアでローカルに保存する方法
-  ベクトルストアから埋め込みモデルをロードし、NVIDIA Endpointsを使用してRAGを構築する概念

## さあ始めましょう

NVIDIA NIMは、[NVIDIA API Catalog](https://build.nvidia.com/explore/discover)で利用可能な、使いやすいオープンAPIを介して素早くアクセスできます。これは、オンラインで幅広いマイクロサービスにアクセスするためのプラットフォームです。NVIDIA NIMを使い始めるには、登録が必要な`NVIDIA API Key`が必要です。以下のスクリーンショットに示すように、`ログインボタンをクリックしてメールアドレスを入力し`、プロセスに従って残りの情報登録するか、[NVIDIA NGC](https://ngc.nvidia.com/signin)の登録(*アカウント名をクリック -> セットアップ -> パーソナルキーの生成*)からAPIキーを生成してください。プロセスが完了したら、API Keyを将来の使用のためにアクセスできる場所に保存してください。APIキーのサンプルは、`nvapi-`で始まり、アンダースコア `_` を含む64文字です。

すでにアカウントをお持ちの方は、この手順に従って`NVIDIA API KEY`を取得してください：

- [ここ](https://build.nvidia.com/explore/discover)からあなたのアカウントでログイン.
- ご希望のモデルをクリックしてください。
- [Input]で[Python]タブを選択し、[Get API Key]をクリックし、[Generate Key]をクリックします。
- 生成されたキーをコピーし、NVIDIA_API_KEYとして保存します。そこからエンドポイントにアクセスできるはずです。

<div style="text-align: center;">
  <!--<img src="imgs/builder_catalog.jpg" style="width: 800px; height: auto;">-->
  <img src="imgs/nim-catalog.png" style="width: 900px; height: auto;">
</div>


### NVIDIA API キーの設定

このNotebookの要件として、NVIDIA NIM の docker イメージを取得する為に、環境変数 `NVIDIA_API_KEY` としてキーを設定する必要があります。まだキーを取得していない場合は、NVIDIA NIM API [ホームページ](https://build.nvidia.com/explore/discover) にアクセスしてAPIキーを生成してください。以下のセルを実行し、表示されるテキストボックスにNVIDIA API KEYを入力し、キーボードのEnterキーを押してください。

In [1]:
import os
import getpass

if not os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    nvapi_key = getpass.getpass("Enter your NVIDIA API key: ")
    assert nvapi_key.startswith("nvapi-"), f"{nvapi_key[:5]}... is not a valid key"
    os.environ["NVIDIA_API_KEY"] = nvapi_key
    os.environ["NGC_API_KEY"] = nvapi_key


Enter your NVIDIA API key:  ········


### LLMとEmbeddingモデルをセルフホスト型NIMをGKEで用意する

### LLMモデルのセットアップ

以下のセルを実行して、NIM LLMサービスを実行するKubernetesサービスが正常に起動していることを確認してください。

In [ ]:
!kubectl get pods

**想定される出力**

No resources found in default namespace.

### 名前空間 (namespace) の作成

NIMのエンドポイントデプロイの用いる名前空間を作成します

In [ ]:
!kubectl create namespace nim-llm

### Secretの作成

KubernetesクラスタがNVCR (NVIDIA Container Registry) からイメージをプルし、NIMアプリケーションがNGC APIキーを利用できるように、必要なSecretを作成します。Documentに記載があるように2種類のsecretを作成します。

[Deploying with Helm](https://docs.nvidia.com/nim/large-language-models/latest/deploy-helm.html)

In [ ]:
!kubectl create secret docker-registry ngc-secret \
  --docker-server=nvcr.io \
  --docker-username='$oauthtoken' \
  --docker-password="${NVIDIA_API_KEY}" \
  --namespace=nim-llm

In [ ]:
!kubectl create secret generic ngc-api \
  --from-literal=NGC_API_KEY="$NVIDIA_API_KEY" \
  -n nim-llm

### NVIDIA NIMの選択

[NIMs Catalog](https://build.nvidia.com/explore/reasoning)には、さまざまなドメインにおける複数の最先端モデルが掲載されています。下のスクリーンショットのように、`RUN ANYWHERE`タグの付いたものを探してください。これらのNIMイメージはダウンロード可能で、モデルや必要な最適化ランタイムを含んでおり、すぐに使い始めるのに役立ちます。

<img src="imgs/catalog1.jpg" style="width: 900px; height: auto;">

お好みのNIMモデルを選択し、dockerタブをクリックし、以下のスクリーンショットのように赤枠内のイメージ名をコピーします。 

<img src="imgs/catalog2.jpg" style="width: 900px; height: auto;">

### NIM LLM Helmチャートのダウンロード

ここでは、NIM LLMをKubernetesにデプロイするための設定パッケージ（Helmチャート）をNVIDIAからダウンロードします。この後のセルで実行するコマンドは、認証を行いつつ、指定したバージョンのHelmチャートファイル (.tgz) を手元に取得します。


In [ ]:
!helm fetch "https://helm.ngc.nvidia.com/nim/charts/nim-llm-1.7.0.tgz" \
    --username='$oauthtoken' \
    --password=$NGC_API_KEY

### Endpointデプロイ用のyamlファイルの作成

次のステップはKubernetesでNIMエンドポイントをデプロイするための設定ファイル (nim_llm.yaml)を作成します。ここでは、`llama-3.1-swallow-8b-instruct-v0.1`をプルすることでこのステップを実演します。

以下のセルを実行するとこのノートブックと同じフォルダにnim_llm.yamlが作成されます。

In [35]:
%%writefile nim_llm_values.yaml
image:
  repository: nvcr.io/nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1
  tag: 1.3.2

imagePullSecrets:
  - name: ngc-secret

model:
  ngcAPISecret: ngc-api
  nimCache: /nim-cache 

persistence:
  enabled: true
  size: "50Gi"  

startupProbe:
  initialDelaySeconds: 300  
  periodSeconds: 60         
  failureThreshold: 360      

resources:
  limits:
    nvidia.com/gpu: 1
    memory: "64Gi"  
    # cpu: "7"        
  requests:
    nvidia.com/gpu: 1
    memory: "32Gi"  
    # cpu: "4"

env:
  - name: NIM_LOW_MEMORY_MODE
    value: "1"
  - name: NIM_MAX_MODEL_LEN
    value: "128"

#extraVolumes:
#  - name: dshm
#    emptyDir:
#      medium: Memory
#extraVolumeMounts:
#  - name: dshm
#    mountPath: /dev/shm



Overwriting nim_llm_values.yaml


### prep. L4用のNIM profileの確認

In [ ]:
! docker run -it --rm  \
    --gpus all  \
    --shm-size=16GB \
    -e NGC_API_KEY  \
    -v "$LOCAL_NIM_CACHE:/opt/nim/.cache"   \
    -u $(id -u)     -p 8000:8000  \
    nvcr.io/nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1:1.3.2

### NIM LLM マイクロサービスの起動

以下のセルを実行し docker run コマンドを実行して NIM LLM マイクロサービスを起動します。


In [ ]:
! minikube status

In [ ]:
! helm template nim-llm-release nim-llm-1.7.0.tgz -n nim-llm > rendered.yaml

In [ ]:
!helm install nim-llm-release ./nim-llm-1.7.0.tgz \
  -f nim_llm_values.yaml \
  --namespace nim-llm \
  --create-namespace

In [25]:
! helm list -n nim-llm

NAME	NAMESPACE	REVISION	UPDATED	STATUS	CHART	APP VERSION


In [ ]:
! kubectl get events -n nim-llm --sort-by=.lastTimestamp

In [ ]:
! helm uninstall nim-llm-release --namespace nim-llm

In [ ]:
!kubectl delete pods -n nim-llm nim-llm-release-0

In [ ]:
! kubectl get all -n nim-llm

In [ ]:
!kubectl get pods -n nim-llm

In [ ]:
! kubectl top pod -n nim-llm

In [ ]:
!kubectl describe pods -n nim-llm nim-llm-release-0

In [ ]:
! kubectl logs -n nim-llm nim-llm-release-0 

In [ ]:
! kubectl describe node | grep -A5 Capacity

In [ ]:
!kubectl get pv

In [ ]:
!kubectl logs nim-llm-release-0 -n nim-llm

**想定される出力**

```
===========================================
== NVIDIA Inference Microservice LLM NIM ==
===========================================

NVIDIA Inference Microservice LLM NIM Version 1.3.0
Model: tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1

Container image Copyright (c) 2016-2024, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

The NIM container is governed by the NVIDIA Software License Agreement (https://www.nvidia.com/en-us/agreements/enterprise-software/nvidia-software-license-agreement/) and Product-Specific Terms for AI Products (https://www.nvidia.com/en-us/agreements/enterprise-software/product-specific-terms-for-ai-products/)


A copy of this license can be found under /opt/nim/LICENSE.


The use of this model is governed by the NVIDIA AI Foundation Models Community License (https://www.nvidia.com/en-us/agreements/enterprise-software/nvidia-ai-foundation-models-community-license-agreement/).


ADDITIONAL INFORMATION: (a) Llama 3.1 Community License Agreement (https://www.llama.com/llama3_1/license/). Built with Llama; and (b) Gemma Terms of Use | Google AI for Developers (https://ai.google.dev/gemma/terms) and Gemma Prohibited Use Policy | Google AI for Developers (https://ai.google.dev/gemma/prohibited_use_policy).



{"level": "None", "time": "None", "file_name": "None", "file_path": "None", "line_number": "-1", "message": "You are using a deprecated `pynvml` package. Please install `nvidia-ml-py` instead. See https://pypi.org/project/pynvml for more information.", "exc_info": "None", "stack_info": "None"}
{"level": "None", "time": "None", "file_name": "None", "file_path": "None", "line_number": "-1", "message": "Profile 375dc0ff86133c2a423fbe9ef46d8fdf12d6403b3caa3b8e70d7851a89fc90dd is not fully defined with checksums", "exc_info": "None", "stack_info": "None"}
...
{"level": "INFO", "time": "2025-05-27 09:57:39.923", "file_name": "httptools_impl.py", "file_path": "/opt/nim/llm/.venv/lib/python3.10/site-packages/uvicorn/protocols/http/httptools_impl.py", "line_number": "481", "message": "127.0.0.1:47306 - \"POST /v1/chat/completions HTTP/1.1\" 200", "exc_info": "None", "stack_info": "None"}
{"level": "INFO", "time": "None", "file_name": "None", "file_path": "None", "line_number": "-1", "message": "[TensorRT-LLM][INFO] Ignoring already terminated request 3", "exc_info": "None", "stack_info": "None"}
{"level": "INFO", "time": "2025-05-27 09:57:47.415", "file_name": "httptools_impl.py", "file_path": "/opt/nim/llm/.venv/lib/python3.10/site-packages/uvicorn/protocols/http/httptools_impl.py", "line_number": "481", "message": "10.244.0.1:49976 - \"GET /v1/health/live HTTP/1.1\" 200", "exc_info": "None", "stack_info": "None"}
{"level": "INFO", "time": "2025-05-27 09:57:47.418", "file_name": "httptools_impl.py", "file_path": "/opt/nim/llm/.venv/lib/python3.10/site-packages/uvicorn/protocols/http/httptools_impl.py", "line_number": "481", "message": "10.244.0.1:49988 - \"GET /v1/health/ready HTTP/1.1\" 200", "exc_info": "None", "stack_info": "None"}
```

しばらく待つとPodがREADY (1/1) になります。

In [ ]:
!kubectl get pods -n nim-llm

**想定する出力**

NAME                READY   STATUS    RESTARTS   AGE
nim-llm-release-0   1/1     Running   0          2m18s

### ポートフォワーディング

システムは複数のプロセスを実行することができるので、実行中のアプリケーションでポートを占有しないようにしなければなりません。以下のコードでは、一意の空きポートを見つけて RAGアプリケーションに割り当てています：

In [ ]:
import random
import socket

def find_available_port(start=11000, end=11999):
    while True:
        # Randomly select a port between start and end range
        port = random.randint(start, end)
        
        # Try to create a socket and bind to the port
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            try:
                sock.bind(("localhost", port))
                # If binding is successful, the port is free
                return port
            except OSError:
                # If binding fails, the port is in use, continue to the next iteration
                continue

# Find and print an available port
os.environ['CONTAINER_PORT'] = str(find_available_port())
print(f"Your have been alloted the available port: {os.environ['CONTAINER_PORT']}")

kubectlのport-forwardコマンドをバックグラウンドで実行するためにsubprocessを使っています。（ターミナル上から実行しても構いません）

In [ ]:
import subprocess
subprocess.Popen(f"kubectl port-forward svc/nim-llm-release {os.environ['CONTAINER_PORT']}:8000 \
                 -n nim-llm".split(), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

割り当てたポートを利用してデプロイしたNIM エンドポイント (Pod)にリクエストを投げてみましょう

In [ ]:
!curl -s -X POST "http://localhost:${CONTAINER_PORT}/v1/chat/completions" \
    -H "accept: application/json" \
    -H "Content-Type: application/json" \
    -d '{ \
        "model": "tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1", \
        "messages": [ \
            { \
                "role": "user", \
                "content": "日本の首都はどこですか？" \
            } \
        ], \
        "temperature": 0.7, \
        "max_tokens": 50 \
    }' \
    | jq

**想定される出力**

```json
{
  "id": "chat-5e8dd5f3d9084596b04af337be67ab1e",
  "object": "chat.completion",
  "created": 1748341341,
  "model": "tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "東京です。"
      },
      "logprobs": null,
      "finish_reason": "stop",
      "stop_reason": null
    }
  ],
  "usage": {
    "prompt_tokens": 18,
    "total_tokens": 21,
    "completion_tokens": 3
  },
  "prompt_logprobs": null
}
```

### クイックテストの開始
NIMが稼働していることを2つの方法で素早くテストできます:
- LangChain NVIDIA Endpoints
- 単純なOpenAI完了リクエスト

**パラメータの説明:**
- **base_url**:  NVIDIA NIM の docker イメージがデプロイされているURL
- **model**: デプロイされた NVIDIA NIM モデルの名前
- **temperature**: サンプリングのランダム性を調整する。温度を下げると、高い確率で単語が選択される可能性が高くなります。
- **top_p**: モデルがどの程度決定論的かを制御します。正確で事実に基づいた回答を求める場合は、この値を低く保ちます。より多様な回答を求める場合は、値を大きくします
- **max_tokens**: 生成される出力トークンの最大数


In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(base_url="http://0.0.0.0:{}/v1".format(os.environ['CONTAINER_PORT']), model="tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1", temperature=0.1, max_tokens=1000, top_p=1.0)

result = llm.invoke("フランスの首都はどこですか？")
print(result.content)

エラーが出力された場合は、しばらく待ってから上記のセルを再実行してください。エラーは、NIMコンテナが完全に立ち上がっていないことが原因かもしれません。

### Embeddingモデルのセットアップ
Llama3.1 Swallow 8Bと同様Embeddingモデルについてもセットアップします。 
以下のセルを実行して、Embedding ModelをGKE経由でデプロイします。

In [4]:
!./scripts/setup_nvembedding_gke.sh

please modify this script for installation nv-embedding nim


Embeddingモデルのインストールが正しく行われたか以下のセルを実行してテストします。

In [ ]:
#https://build.nvidia.com/nvidia/llama-3_2-nv-embedqa-1b-v2/
curl -X "POST" \
  "http://localhost::${EMBED_CONTAINER_PORT}/v1/embeddings" \
  -H 'accept: application/json' \
  -H 'Content-Type: application/json' \
  -d '{
    "input": ["NVIDIAの新しく出たGPUには何がありますか？名前を教えてください。"],
    "model": "nvidia/llama-3.2-nv-embedqa-1b-v2",
    "input_type": "query"
}'

## RAG(検索拡張生成)

### RAGとは?

RAG（検索拡張生成）は、外部ソースから取得した事実を用いて生成AIモデルの精度と信頼性を向上させる技術です。
事前に学習された大規模な言語モデル(LLM)は、そのパラメータに事実知識を格納することが示されており、ダウンストリームの自然言語処理タスクでファインチューニングを行うと、最先端の結果を達成します。しかし、知識にアクセスして正確に操作する能力はまだ限定的であるため、知識集約的なタスクでは、タスクに特化したアーキテクチャに比べて性能が劣ります。さらに、その決定に実証性を与え、世界知識を更新することは、依然として未解決の研究課題です。詳しくは[こちら](https://arxiv.org/pdf/2005.11401)をご確認下さい。 
 

#### RAGはどのように役立つのか？
RAGは、大規模言語モデル（LLM）の強力なセマンティック機能と、メタデータ、テキスト、画像、ビデオ、表、グラフなどを含むデータセットとの接続を支援するアーキテクチャです。

<div style="text-align: center;">
    <img src="https://www.nvidia.com/en-in/glossary/retrieval-augmented-generation/_jcr_content/root/responsivegrid/nv_container_6840787/nv_image.coreimg.100.630.png/1714589610702/rag-diagram-1920x1080.png" alt="RAG Architecture" style="width: 80%; max-width: 600px;"> <br>
    RAG Architecture
</div>

### RAG コンポーネント

RAGアプリケーションは`Retriever` と ` Generator`の2つの主要コンポーネントから構成されています: . `Retriever` (RET) コンポーネントは、ユーザからの問い合わせに応答して、保存されているデータセットから適切な情報を抽出します。続いて、` Generator`(GEN)コンポーネントは、ユーザーからの問い合わせと検索された情報を利用して応答を生成します。

RAGシステムのアーキテクチャは、主に以下の要素で構成されています:
- **RET**
    - 埋め込みモデル: 入力データを密なベクトル表現に変換するバイエンコーダーネットワーク
    - ベクトルストア: 高次元ベクトル埋め込みを格納し、クエリするために最適化されたデータベース.
    - *リ*-ランキング (オプショナル): クエリとの関連性に基づいて検索された情報に優先順位を付けるクロスエンコーダ

- **GEN**
    - 大規模言語モデル (LLM): 膨大なテキストデータで訓練されたニューラルネットワークで、人間のような応答を生成することができます。
    - ガードレール (オプショナル): 生成された出力が事前に定義された制約や品質基準に準拠していることを保証するために実装されたメカニズムです。

堅牢なRAGアプリケーションには、以下に関して優れている必要があります：
- `Retriever`パイプライン
- ` Generator`モデル
- 両者のリンク

ラボでは、これらのコンポーネントを1つずつインスタンス化し、それらをまとめます。

## LangChainの概要

[LangChain](https://python.langchain.com/v0.2/docs/introduction/)は、大規模言語モデル(LLM)を使用するアプリケーションの開発を簡素化するために設計された強力なフレームワークです。
RAGアプリケーションのために、LangChainは以下を提供します:

- 様々なファイルタイプのドキュメントローダー
- ドキュメントを分割するテキスト分割機能
- テキストをベクトル表現に変換する埋め込み
- 効率的な類似検索のためのベクトルストア
- 関連情報を取得するための`Retriever`メソッド
- クエリと応答を構造化するプロンプトテンプレート

### RAG アプリケーション

このセクションでは、前のノートブックの手順に従って、ローカルに配置されたNVIDIA NIMをベースとしたRAGアプリケーションを構築します。このデモでは、前のノートブックのように2つのLLMを使って会話型検索チェーンを作るのではなく、1つのLLM `llama-3.1-swallow--8b-instruct` を使って会話型検索チェーンを作ります。これは各NIMイメージには1つのベースモデルがあるからです。ローカルに配置されたNIMとリモートアクセスを使用することも可能ですが、わかりやすくするために、単一のLLMのアプローチにこだわります。 

#### ライブラリのインポート

In [ ]:
from langchain.chains import ConversationalRetrievalChain, LLMChain
from langchain.chains.conversational_retrieval.prompts import CONDENSE_QUESTION_PROMPT, QA_PROMPT
from langchain.chains.question_answering import load_qa_chain
from langchain.memory import ConversationBufferMemory
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

#### ウェブリンク・データソースの作成

お好みのウェブリンクを差し替えたり、追加したりすることができます。

In [ ]:
urls = [
"https://www.fsa.go.jp/news/r5/20230829/230829_main.pdf",
"https://www.tioj.or.jp/activity/pdf/190619_06.pdf",
"https://www.pmda.go.jp/files/000251332.pdf",
"https://www.jili.or.jp/files/research/zenkokujittai/pdf/r3/i-xvii.pdf",
"https://www.zenginkyo.or.jp/fileadmin/res/news/news350331_1.pdf"
]


## 実装

#### PDF ファイルを読み込む関数を作る

以下は、PDFファイルを読み込むためのヘルパー関数です。

In [ ]:
from pypdf import PdfReader
from io import BytesIO
import requests
import re

def clean_japanese_pdf_text(text: str) -> str:
    # 日本語文中の改行を削除
    text = re.sub(r'(?<=[ぁ-んァ-ヶ一-龥])\s*\n\s*(?=[ぁ-んァ-ヶ一-龥])', '', text)
    # 連続するスペースや全角スペースを1つにまとめる
    text = re.sub(r'[ 　]{2,}', ' ', text)
    # 各行の先頭空白を削除
    text = re.sub(r'^\s+', '', text, flags=re.MULTILINE)
    # 「..... 4」形式の目次行のページ番号を除去
    text = re.sub(r'\.{3,}\s*\d{1,3}', '', text)
    text = text.replace("\n", "")
    return text

def pdf_document_loader(url: str) -> str:
    """
    Loads and extracts cleaned text from a PDF at the given URL using `pypdf`.

    Args:
        url: The URL of the PDF.

    Returns:
        Cleaned text content of the PDF.
    """
    try:
        response = requests.get(url)
        response.raise_for_status()
        reader = PdfReader(BytesIO(response.content))
        text = ""
        for page in reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted
        return clean_japanese_pdf_text(text.strip())
    except Exception as e:
        print(f"Failed to load {url} due to: {e}")
        return ""



#### 埋め込みとDocument Text Splitterの作成

埋め込みを保存するパスを初期化し、`pdf_document_loader`関数を実行し、ドキュメントをテキストの塊に分割する関数を作ってみましょう。

In [ ]:
from tqdm import tqdm

def create_embeddings(embeddings_model,embedding_path: str = "./embed"):

    embedding_path = "./embed"
    print(f"Storing embeddings to {embedding_path}")

    documents = []
    for url in tqdm(urls):
        print(url)
        document = pdf_document_loader(url)
        #document = html_document_loader(url)
        documents.append(document)


    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=0,
        length_function=len,
    )
    print("Total documents:",len(documents))
    texts = text_splitter.create_documents(documents)
    print("Total texts:",len(texts))
    index_docs(embeddings_model,url, text_splitter, texts, embedding_path,)
    print("Generated embedding successfully")

#### [Need to update]LangChainからNVIDIA AI Endpointsを使って埋め込みを生成する

このセクションでは、LangChain用のNVIDIA AI Endpointsを使って埋め込みを生成し、将来の再利用のためにエンベッディングを`/embed`ディレクトリのオフラインベクタストアに保存する方法をデモします。

In [ ]:
embeddings_model = NVIDIAEmbeddings(model="NV-Embed-QA") # or use nvidia/nv-embedqa-e5-v5

以下では、ドキュメントページのコンテンツをループしてテキストとメタデータを拡張し、[FAISS](https://faiss.ai/index.html)を適用する `index_docs` 関数を作成します。埋め込みはローカルに保存されます。

In [ ]:
from typing import List, Union


def index_docs(embeddings_model, url: Union[str, bytes], splitter, documents: List[str], dest_embed_dir: str) -> None:
    """
    Split the documents into chunks and create embeddings for them.
    
    Args:
        embeddings_model: Model used for creating embeddings.
        url: Source url for the documents.
        splitter: Splitter used to split the documents.
        documents: List of documents whose embeddings need to be created.
        dest_embed_dir: Destination directory for embeddings.
    """
    texts = []
    metadatas = []

    for document in documents:
        chunk_texts = splitter.split_text(document.page_content)
        texts.extend(chunk_texts)
        metadatas.extend([document.metadata] * len(chunk_texts))

    if os.path.exists(dest_embed_dir):
        docsearch = FAISS.load_local(
            folder_path=dest_embed_dir, 
            embeddings=embeddings_model, 
            allow_dangerous_deserialization=True
        )
        docsearch.add_texts(texts, metadatas=metadatas)
    else:
        docsearch = FAISS.from_texts(texts, embedding=embeddings_model, metadatas=metadatas)

    docsearch.save_local(folder_path=dest_embed_dir)

#### ベクターストアからの埋め込みのロードとNVIDIA Endpointを使用したRAGの構築

次に、関数 `create_embeddings` を呼び出し、FAISSを使って[vector store](https://developer.nvidia.com/blog/accelerating-vector-search-fine-tuning-gpu-index-algorithms/)から文書を読み込む。ベクトルストアはembeddingsと呼ばれる高次元空間に関連情報を格納しています。

以下の2つのセルを実行してください。

In [ ]:
%%time
create_embeddings(embeddings_model=embeddings_model)

In [ ]:
# load Embed documents
embedding_path = "./embed/"
docsearch = FAISS.load_local(folder_path=embedding_path, embeddings=embeddings_model, allow_dangerous_deserialization=True)

### llama-3.1-swallow-8b-instructで会話型検索チェーンを作る

デプロイされたNIMは`http://0.0.0.0:8000`で稼働しているので、NIMの基本モデル`llama-3.1-swallow-8b-instruct`に基づいて[会話型検索チェーン](https://python.langchain.com/v0.1/docs/modules/chains/#conversationalretrievalchain-with-streaming-to-stdout)を作成する。

In [ ]:
llm = ChatNVIDIA(base_url="http://0.0.0.0:{}/v1".format(os.environ['CONTAINER_PORT']),
                 model="tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1", temperature=0.0, max_tokens=1000, top_p=1.0)

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

qa_prompt=QA_PROMPT

doc_chain = load_qa_chain(llm, chain_type="stuff", prompt=QA_PROMPT)

qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=docsearch.as_retriever(),
    chain_type="stuff",
    memory=memory,
    combine_docs_chain_kwargs={'prompt': qa_prompt},
)

### クエリによるテスト

In [ ]:
query = "世帯主が不意の事故により入院が必要になる場合の必要資金について、60～64歳及び65歳以上の夫婦が公的年金以 \
        外に必要とする月間生活資金と比較してください。"
result = qa({"question": query})
print(result.get("answer"))

In [ ]:
query = "製造たばこの個包装及び中間包装に求められる識別マークの表示方法の条件について説明してください。"
result = qa({"question": query})
print(result.get("answer"))

お疲れ様でした。

---

## References

- https://developer.nvidia.com/blog/tips-for-building-a-rag-pipeline-with-nvidia-ai-langchain-ai-endpoints/
- https://nvidia.github.io/GenerativeAIExamples/latest/notebooks/05_RAG_for_HTML_docs_with_Langchain_NVIDIA_AI_Endpoints.html

## Licensing

Copyright © 2024 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.

<br>
<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="rag_nim_endpoints.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="rag_nim_endpoints.ipynb">1</a>
        <a >2</a>
        <a href="nim_lora_adapter.ipynb">3</a>
        <!-- <a href="challenge.ipynb">4</a> -->
    </span>
    <span style="float: left; width: 33%; text-align: right;"><a href="nim_lora_adapter.ipynb">Next Notebook</a></span>
</div>

<br>
<p> <center> <a href="../../Start-NIM-RAG.ipynb">Home Page</a> </center> </p>